In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn
!pip install openpyxl xlrd


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")


In [ ]:
import pandas as pd

# Load datasets with specified encoding
customer_data = pd.read_csv(r"D:\PROJECTS\EDA\DATA'S\Customers.csv",encoding='ISO-8859-1')
data_dictionary_df = pd.read_csv(r"D:\PROJECTS\EDA\DATA'S\Data_Dictionary.csv", encoding='ISO-8859-1')
exchange_rates_df = pd.read_csv(r"D:\PROJECTS\EDA\DATA'S\Exchange_Rates.csv", encoding='ISO-8859-1')
products_df = pd.read_csv(r"D:\PROJECTS\EDA\DATA'S\Products.csv", encoding='ISO-8859-1')
sales_df = pd.read_csv(r"D:\PROJECTS\EDA\DATA'S\Sales.csv", encoding='ISO-8859-1')
stores_df = pd.read_csv(r"D:\PROJECTS\EDA\DATA'S\Stores.csv", encoding='ISO-8859-1')

# Display the first few rows of each DataFrame to check
print(customer_data.head())
print(data_dictionary_df.head())
print(exchange_rates_df.head())
print(products_df.head())
print(sales_df.head())
print(stores_df.head())


In [ ]:
# Check for missing values
print("Missing values in each column:")
print(customer_data.isnull().sum())

In [ ]:
            # HANDLING MISSING VALUES
# Calculate the mode for 'State Code'
most_common_state_code = customer_data['State Code'].mode()[0]

# Fill missing values with the mode by assigning directly
customer_data['State Code'] = customer_data['State Code'].fillna(most_common_state_code)

# Verify that there are no more missing values
print("Missing values after imputation:")
print(customer_data.isnull().sum())


In [ ]:
# Check data types of each column
print(customer_data.dtypes)


In [ ]:
# Convert 'Birthday' to datetime and calculate 'Age'
customer_data['Birthday'] = pd.to_datetime(customer_data['Birthday'], errors='coerce')
customer_data['Age'] = (pd.to_datetime('today') - customer_data['Birthday']).dt.days // 365
print(customer_data.dtypes)

In [ ]:
            # Demographic Distribution
# Gender distribution
gender_distribution = customer_data['Gender'].value_counts()
print("Gender Distribution:")
print(gender_distribution)


In [ ]:
# Age Distribution
import matplotlib.pyplot as plt

# Age distribution
plt.figure(figsize=(10, 6))
plt.hist(customer_data['Age'], bins=20, color='skyblue', edgecolor='black')
plt.title('Age Distribution of Customers')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.grid(axis='y')
plt.show()


In [ ]:
#  Location Distribution
# State distribution
state_distribution = customer_data['State'].value_counts()
print("State Distribution:")
print(state_distribution)

# Country distribution
country_distribution = customer_data['Country'].value_counts()
print("Country Distribution:")
print(country_distribution)


In [ ]:
# Merge sales data with customer data to create merged_df
merged_df = sales_df.merge(customer_data[['CustomerKey', 'Gender', 'Birthday', 'Age']], on='CustomerKey', how='left')
print(merged_df)

In [ ]:
# Convert 'Order Date' to datetime
merged_df['Order Date'] = pd.to_datetime(merged_df['Order Date'])

# Calculate Total Sales and Number of Orders per Customer
purchase_summary = merged_df.groupby('CustomerKey').agg(
    Total_Sales=('Quantity', 'sum'),
    Number_of_Orders=('Order Number', 'nunique')
).reset_index()


In [ ]:
# Calculate Average Order Value (AOV)
purchase_summary['Average_Order_Value'] = purchase_summary['Total_Sales'] / purchase_summary['Number_of_Orders']

# Define segments based on Average Order Value
def segment_customers(row):
    if row['Average_Order_Value'] < 10:
        return 'Low Value'
    elif row['Average_Order_Value'] < 20:
        return 'Medium Value'
    else:
        return 'High Value'

# Apply the segmentation function
purchase_summary['Customer_Segment'] = purchase_summary.apply(segment_customers, axis=1)

# Display the segmented customer summary
print(purchase_summary[['CustomerKey', 'Total_Sales', 'Number_of_Orders', 'Average_Order_Value', 'Customer_Segment']].head())

In [ ]:
# Bar plot for customer segments
plt.figure(figsize=(8, 5))
segment_counts = purchase_summary['Customer_Segment'].value_counts()
sns.barplot(x=segment_counts.index, y=segment_counts.values, palette='viridis', hue=None)
plt.title('Customer Segmentation by Average Order Value')
plt.xlabel('Customer Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Merge Age information back into purchase_summary
purchase_summary = purchase_summary.merge(merged_df[['CustomerKey', 'Age']], on='CustomerKey', how='left')

# Calculate average age by customer segment
average_age_by_segment = purchase_summary.groupby('Customer_Segment')['Age'].mean().reset_index()

# Display average age by segment
print(average_age_by_segment)
#Completed Customer Analysis

In [ ]:
# Define the file path
output_path = r"D:\PROJECTS\EDA\Cleanned Data\customer_analysis.csv"

# Save the purchase_summary DataFrame to an Excel file
purchase_summary.to_csv(output_path, index=False)


In [ ]:
pip install pandas sqlalchemy pymysql


In [ ]:
from sqlalchemy import create_engine

# Database connection details
user = 'root'
password = 'Krishna3_6_9'
host = '127.0.0.1'
database = 'Datas'

# Create a database connection
connection_string = f'mysql+pymysql://{user}:{password}@{host}/{database}'
engine = create_engine(connection_string)

# Store the purchase_summary DataFrame to the database
purchase_summary.to_sql('Customer_analysis', con=engine, index=False, if_exists='replace')

print("Data successfully stored in the MySQL database.")


In [ ]:
purchase_summary.dtypes